In [1]:
# check .ipynb file run properly
a = 1
a

1

In [2]:
import os 

In [3]:
%pwd    # it is said about current location in local system 

'c:\\Users\\deven\\Desktop\\Projects\\DLops\\research'

In [4]:
os.chdir("../") # it is used to go backward from current dir

In [5]:
%pwd

'c:\\Users\\deven\\Desktop\\Projects\\DLops'

In [8]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
# dataclass: saves time by creating basic functions automatically
# frozen=True: makes values fixed (you cannot change them later)
# Use case: prevents accidental changes in important settings/configuration
class DataIngestionConfig:
    """
    This class stores all settings needed to download and extract data.

    Use cases (simple):
    - Keeps all paths and links in one place
    - Makes code clean and easy to manage
    - Prevents accidental changes (because frozen=True)
    """

    root_dir: Path
    # Main folder where all data files will be kept
    # Example: "artifacts/data_ingestion/"

    source_URL: str
    # Link from where the data will be downloaded
    # Example: a zip file link from internet

    local_data_file: Path
    # Where the downloaded file will be saved
    # Example: "data.zip"

    unzip_dir: Path
    # Folder where the zip file will be extracted
    # Example: "unzipped_data/"

In [7]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [11]:
import os
import urllib.request as request
import zipfile
from src.cnnClassifier import logger
from src.cnnClassifier.utils.common import get_size

In [12]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  


    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [14]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
     raise e

✅ YAML Content: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://raw.githubusercontent.com/1602saurab/data_im/main/Chicken-fecal-images%20(2).zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}}
[2026-04-25 10:40:12,214: INFO: common: YAML file config\config.yaml loaded successfully]
✅ YAML Content: {'AUGMENTATION': True, 'IMAGE_SIZE': [224, 224, 3], 'BATCH_SIZE': 16, 'INCLUDE_TOP': False, 'EPOCHS': 1, 'CLASSES': 2, 'WEIGHTS': 'imagenet', 'LEARNING_RATE': 0.01}
[2026-04-25 10:40:12,216: INFO: common: YAML file params.yaml loaded successfully]
[2026-04-25 10:40:12,216: INFO: common: created directory at: artifacts]
[2026-04-25 10:40:12,216: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-25 10:40:12,751: INFO: 4037744573: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-ag